# 🧪 Notebook 07 — Hyperparameter Tuning

## 🎯 Objective
Systematically search for better model configurations using cross-validated hyperparameter tuning
to maximize **ROC-AUC** for heart-disease prediction.

This notebook builds on:
- `02_data_loading_preprocessing.ipynb` — cleaned & split data
- `04_feature_engineering.ipynb` — feature engineering + preprocessing
- `05_model_training.ipynb` — baseline models and selection

We’ll run **RandomizedSearchCV / GridSearchCV** over candidate spaces for:
- Logistic Regression
- Random Forest
- (Optional) XGBoost

Then we’ll:
1. Pick the **best validation model**,
2. Refit it on train+val,
3. Evaluate on the **test** set,
4. Save artifacts and an updated `reports/best_model.json`.

---

## ⚙️ Workflow
1. Load processed splits (train/val/test)
2. Build (or load) the preprocessing pipeline (`src/features.py`)
3. Define search spaces per model
4. Run randomized/grid CV with **StratifiedKFold**
5. Rank candidates by CV ROC-AUC and validate on the holdout validation set
6. Evaluate the winner on test + save metrics/artifacts
7. Persist CV results for reproducibility

---

## 📦 Outputs
- `models/heart_model_<run_id>.pkl` — tuned pipeline (preprocess + model)
- `reports/val_metrics_<run_id>.json` — validation metrics
- `reports/test_metrics_<run_id>.json` — test metrics
- `reports/hparam_search_<model>_<timestamp>.csv` — CV results table
- `reports/best_model.json` — updated pointer to the best run

---


## 0) Setup and imports

In [2]:
# (Colab only) mount & cd
IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Dr. Taiwo famuyiwa - Data Science & Biostatistics Portfolio/Machine Learning Projects/Heart-Disease-Prediction



Mounted at /content/drive
/content/drive/MyDrive/Dr. Taiwo famuyiwa - Data Science & Biostatistics Portfolio/Machine Learning Projects/Heart-Disease-Prediction


In [3]:
# std
import os, sys, json, time, warnings
warnings.filterwarnings("ignore")
sys.path.append(os.getcwd())

import numpy as np
import pandas as pd
from pathlib import Path

# sklearn
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, precision_score, recall_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Optional XGBoost
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

# project modules
from src.data import PROC_DIR
from src.features import build_preprocessor

# Paths
MODELS_DIR = Path("models"); MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("reports"); REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_JOBS = -1

## 1) Load splits

In [4]:
X_train = pd.read_csv(PROC_DIR / "X_train.csv")
y_train = pd.read_csv(PROC_DIR / "y_train.csv").squeeze("columns")

X_val   = pd.read_csv(PROC_DIR / "X_val.csv")
y_val   = pd.read_csv(PROC_DIR / "y_val.csv").squeeze("columns")

X_test  = pd.read_csv(PROC_DIR / "X_test.csv")
y_test  = pd.read_csv(PROC_DIR / "y_test.csv").squeeze("columns")

X_train.shape, X_val.shape, X_test.shape, float(y_train.mean())


((194, 13), (49, 13), (61, 13), 0.4587628865979381)

## 2) Import preprocessor

In [5]:
import joblib
pre = joblib.load("models/preprocessor.pkl")  # fitted

##3) Define models & search spaces

In [6]:
def model_spaces():
    spaces = {}

    # Logistic Regression (L2, class_weight options, C range)
    spaces["logreg"] = {
        "estimator": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "param_distributions": {
            "model__C": np.logspace(-3, 2, 20),
            "model__solver": ["lbfgs", "liblinear"],
            "model__class_weight": [None, "balanced"]
        },
        "type": "random",
        "n_iter": 40
    }

    # Random Forest
    spaces["rf"] = {
        "estimator": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS),
        "param_distributions": {
            "model__n_estimators": [200, 400, 600, 800],
            "model__max_depth": [None, 4, 6, 8, 10, 12],
            "model__min_samples_split": [2, 4, 6, 10],
            "model__min_samples_leaf": [1, 2, 3, 5],
            "model__max_features": ["sqrt", "log2", 0.5, 0.7, 1.0],
            "model__class_weight": [None, "balanced"]
        },
        "type": "random",
        "n_iter": 50
    }

    # XGBoost (optional)
    if HAS_XGB:
        spaces["xgb"] = {
            "estimator": XGBClassifier(
                objective="binary:logistic",
                eval_metric="auc",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=N_JOBS
            ),
            "param_distributions": {
                "model__n_estimators": [300, 500, 800, 1000],
                "model__max_depth": [3, 4, 5, 6],
                "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
                "model__subsample": [0.7, 0.8, 0.9, 1.0],
                "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
                "model__reg_lambda": [0.0, 0.5, 1.0, 2.0, 5.0],
                "model__reg_alpha": [0.0, 0.1, 0.5]
            },
            "type": "random",
            "n_iter": 60
        }

    return spaces

spaces = model_spaces()
list(spaces.keys())


['logreg', 'rf', 'xgb']

##4) Generic tuner helpers

In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
import joblib

def build_pipe(estimator):
    return Pipeline([("pre", pre), ("model", estimator)])

def run_random_search(name, estimator, distributions, X, y, n_iter=50, cv_splits=5, seed=RANDOM_STATE):
    pipe = build_pipe(estimator)
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=seed)
    rs = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=distributions,
        n_iter=n_iter,
        scoring="roc_auc",
        n_jobs=N_JOBS,
        cv=cv,
        verbose=1,
        random_state=seed,
        refit=True  # keep best estimator refit on all training folds
    )
    rs.fit(X, y)
    return rs

def run_grid_search(name, estimator, grid, X, y, cv_splits=5, seed=RANDOM_STATE):
    pipe = build_pipe(estimator)
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=seed)
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="roc_auc",
        n_jobs=N_JOBS,
        cv=cv,
        verbose=1,
        refit=True
    )
    gs.fit(X, y)
    return gs

def dump_cv_results(search, model_name):
    ts = int(time.time())
    df = pd.DataFrame(search.cv_results_)
    out_csv = REPORTS_DIR / f"hparam_search_{model_name}_{ts}.csv"
    df.to_csv(out_csv, index=False)
    print("📝 Wrote CV table:", out_csv)
    return out_csv


## 5) Run searches per model

In [8]:
search_results = {}

for name, spec in spaces.items():
    print(f"\n=== Tuning: {name} ===")
    est = spec["estimator"]
    if spec["type"] == "random":
        search = run_random_search(
            name,
            est,
            spec["param_distributions"],
            X_train, y_train,
            n_iter=spec.get("n_iter", 50),
        )
    else:
        search = run_grid_search(
            name,
            est,
            spec["param_grid"],
            X_train, y_train,
        )

    # Save CV table
    csv_path = dump_cv_results(search, name)

    # Track
    best_auc = search.best_score_
    search_results[name] = {
        "best_auc_cv": float(best_auc),
        "best_params": search.best_params_,
        "csv": str(csv_path),
        "search_obj": search
    }
    print(f"Best CV AUC ({name}): {best_auc:.3f}")



=== Tuning: logreg ===
Fitting 5 folds for each of 40 candidates, totalling 200 fits
📝 Wrote CV table: reports/hparam_search_logreg_1761366399.csv
Best CV AUC (logreg): 0.907

=== Tuning: rf ===
Fitting 5 folds for each of 50 candidates, totalling 250 fits
📝 Wrote CV table: reports/hparam_search_rf_1761366703.csv
Best CV AUC (rf): 0.896

=== Tuning: xgb ===
Fitting 5 folds for each of 60 candidates, totalling 300 fits
📝 Wrote CV table: reports/hparam_search_xgb_1761366767.csv
Best CV AUC (xgb): 0.890


## 6) Validate winners on the validation set & save artifacts

In [9]:
def validate_and_save(name, search_obj, X_va, y_va):
    # best_estimator_ is already a fitted Pipeline(pre + model) on full CV training folds
    best_pipe = search_obj.best_estimator_

    # Refit on full TRAIN set (optionally combine train+val later)
    best_pipe.fit(X_train, y_train)

    # Validate
    proba = best_pipe.predict_proba(X_va)[:, 1]
    val_auc = roc_auc_score(y_va, proba)
    val_ap  = average_precision_score(y_va, proba)

    # Save model
    run_id = f"{name}_tuned_seed{RANDOM_STATE}_{int(time.time())}"
    model_path = MODELS_DIR / f"heart_model_{run_id}.pkl"
    joblib.dump(best_pipe, model_path)

    # Save val metrics
    val_metrics = {"run_id": run_id, "val_auc": float(val_auc), "val_ap": float(val_ap), "best_params": search_obj.best_params_}
    (REPORTS_DIR / f"val_metrics_{run_id}.json").write_text(json.dumps(val_metrics, indent=2))

    print(f"💾 Saved model: {model_path.name}")
    print(f"📝 Val AUC={val_auc:.3f}  AP={val_ap:.3f}")
    return run_id, val_auc, val_ap, model_path

val_rows = []
for name, s in search_results.items():
    run_id, v_auc, v_ap, path = validate_and_save(name, s["search_obj"], X_val, y_val)
    val_rows.append((name, run_id, v_auc, v_ap, path.name))

df_val = pd.DataFrame(val_rows, columns=["model","run_id","val_auc","val_ap","artifact"])
df_val.sort_values(["val_auc","val_ap"], ascending=False)


💾 Saved model: heart_model_logreg_tuned_seed42_1761367619.pkl
📝 Val AUC=0.965  AP=0.966
💾 Saved model: heart_model_rf_tuned_seed42_1761367619.pkl
📝 Val AUC=0.957  AP=0.959
💾 Saved model: heart_model_xgb_tuned_seed42_1761367619.pkl
📝 Val AUC=0.948  AP=0.953


,model,run_id,val_auc,val_ap,artifact
0,logreg,logreg_tuned_seed42_1761367619,0.964883,0.966380,heart_model_logreg_tuned_seed42_1761367619.pkl
1,rf,rf_tuned_seed42_1761367619,0.956522,0.958653,heart_model_rf_tuned_seed42_1761367619.pkl
2,xgb,xgb_tuned_seed42_1761367619,0.948161,0.953221,heart_model_xgb_tuned_seed42_1761367619.pkl


## 7) Pick the best by validation → evaluate on test

In [10]:
best_row = df_val.sort_values(["val_auc","val_ap"], ascending=False).iloc[0]
best_row


,0
model,logreg
run_id,logreg_tuned_seed42_1761367619
val_auc,0.964883
val_ap,0.96638
artifact,heart_model_logreg_tuned_seed42_1761367619.pkl


In [11]:
import joblib

best_model_path = MODELS_DIR / best_row.artifact
pipe_best = joblib.load(best_model_path)

test_proba = pipe_best.predict_proba(X_test)[:, 1]
test_auc   = roc_auc_score(y_test, test_proba)
test_ap    = average_precision_score(y_test, test_proba)

thr = 0.5
test_pred = (test_proba >= thr).astype(int)

metrics = {
    "best_model": best_row.model,
    "run_id": best_row.run_id,
    "test_auc": float(test_auc),
    "test_ap": float(test_ap),
    "accuracy@0.5": float(accuracy_score(y_test, test_pred)),
    "precision@0.5": float(precision_score(y_test, test_pred)),
    "recall@0.5": float(recall_score(y_test, test_pred)),
    "f1@0.5": float(f1_score(y_test, test_pred)),
}
print(json.dumps(metrics, indent=2))
(REPORTS_DIR / f"test_metrics_{best_row.run_id}.json").write_text(json.dumps(metrics, indent=2))


{
  "best_model": "logreg",
  "run_id": "logreg_tuned_seed42_1761367619",
  "test_auc": 0.9080086580086579,
  "test_ap": 0.8730707065492015,
  "accuracy@0.5": 0.8524590163934426,
  "precision@0.5": 0.8275862068965517,
  "recall@0.5": 0.8571428571428571,
  "f1@0.5": 0.8421052631578947
}


286

## 8) Update pointer to “best model”

In [12]:
pointer = {
    "best_run_id": best_row.run_id,
    "model": best_row.model,
    "source": "hyperparameter_tuning_notebook_07"
}
(REPORTS_DIR / "best_model.json").write_text(json.dumps(pointer, indent=2))
print("🔖 Updated:", REPORTS_DIR / "best_model.json")


🔖 Updated: reports/best_model.json


## 9) Refit on train+val for final model

In [14]:
# If you want to train on more data before test:
from sklearn.model_selection import train_test_split

X_trval = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_trval = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

pipe_best.fit(X_trval, y_trval)
refit_run_id = f"{best_row.model}_tuned_trval_seed{RANDOM_STATE}_{int(time.time())}"

import joblib
joblib.dump(pipe_best, MODELS_DIR / f"heart_model_{refit_run_id}.pkl")
print("💾 Refit model on train+val saved as:", f"heart_model_{refit_run_id}.pkl")


💾 Refit model on train+val saved as: heart_model_logreg_tuned_trval_seed42_1761367818.pkl


## 10) Save a compact leaderboard of runs

In [15]:
lb = df_val.copy()
for name, s in search_results.items():
    lb.loc[lb.model == name, "cv_auc_best"] = s["best_auc_cv"]
lb = lb.sort_values(["val_auc","val_ap"], ascending=False)
lb.to_csv(REPORTS_DIR / "leaderboard_tuning.csv", index=False)
lb


,model,run_id,val_auc,val_ap,artifact,cv_auc_best
0,logreg,logreg_tuned_seed42_1761367619,0.964883,0.966380,heart_model_logreg_tuned_seed42_1761367619.pkl,0.907252
1,rf,rf_tuned_seed42_1761367619,0.956522,0.958653,heart_model_rf_tuned_seed42_1761367619.pkl,0.895798
2,xgb,xgb_tuned_seed42_1761367619,0.948161,0.953221,heart_model_xgb_tuned_seed42_1761367619.pkl,0.889698
